In [ ]:
# =============================================================================
# DAY 27 — Vision Transformers (ViT) & CLIP from Scratch
# =============================================================================
# So far in the course:
#   NLP / transformers → attention over TOKENS (words / subwords)
#   CNNs (earlier weeks) → local filters sliding over pixels
#   RL (Days 22–26) → agents & policies
#
# Today: bring transformers to IMAGES, then ALIGN images with TEXT (CLIP).
#
#
# ---------------------------------------------------------------------------
# 1) VISION TRANSFORMER (ViT) — "an image is a sequence of patches"
# ---------------------------------------------------------------------------
#
# Classic CNN: convolution sees a local neighborhood, stacks layers to grow
# receptive field. Strong inductive bias (locality, translation equivariance).
#
# ViT idea (Dosovitskiy et al., 2020): throw away convolutions for the trunk.
# Treat the image like a sentence:
#
#   1. Split image into fixed-size PATCHES (e.g. 16×16)
#      224×224 image → (224/16)² = 14×14 = 196 patches
#
#   2. Flatten each patch → small vector of raw pixels
#      e.g. 16×16×3 = 768 numbers
#
#   3. Linear PROJECT each patch into an embedding of dim D
#      (same role as token embedding in NLP)
#
#   4. Add a learnable [CLS] token at the front (or use mean pooling)
#      [CLS] will aggregate the whole image for classification
#
#   5. Add POSITIONAL embeddings (patches have no order otherwise —
#      transformers are permutation-invariant without them)
#
#   6. Run a standard Transformer ENCODER (self-attention + MLP blocks)
#
#   7. Take [CLS] (or pooled) representation → linear head → class logits
#
# Mental model:
#   CNN  = "look at nearby pixels, build up"
#   ViT  = "cut the photo into stamps, let every stamp talk to every other stamp"
#
# Pros: simple, scales extremely well with data/compute; global context from
#       layer 1 (any patch can attend to any other).
# Cons: needs LOTS of data (or strong pretraining) — weak locality bias vs CNNs
#       on small datasets; quadratic attention cost in number of patches.
#
# Hybrid note: many practical "ViTs" still use a small CNN stem; pure ViT is
# the clean teaching version we care about conceptually.
#
#
# ---------------------------------------------------------------------------
# 2) MULTI-HEAD SELF-ATTENTION (RECAP)
# ---------------------------------------------------------------------------
#
# Given a sequence of vectors X ∈ R^{N×D}  (N = #tokens/patches, D = dim):
#
#   Q = X W_Q ,  K = X W_K ,  V = X W_V
#
#   Attention(Q,K,V) = softmax( Q Kᵀ / √d_k ) V
#
# English:
#   "Each token writes a QUERY ('what am I looking for?'),
#    every token offers a KEY ('what do I contain?'),
#    similarity Q·K → how much to read from that token's VALUE."
#
# √d_k scaling keeps dot products from exploding so softmax stays soft.
#
# MULTI-HEAD: split into H heads, each with its own W_Q/W_K/W_V on a slice
# of the channels. Different heads can learn different relations
# (e.g. one head: color similarity; another: spatial neighbors; another: object parts).
# Concatenate heads → mix with W_O.
#
# In ViT:
#   N ≈ 197  (196 patches + CLS) — manageable
#   Self-attention lets the [CLS] token gather evidence from anywhere in the image
#   (cat ears in corner + whiskers elsewhere → "cat")
#
# Transformer BLOCK (encoder) typically:
#   x ← x + Attention(LN(x))     # residual + LayerNorm
#   x ← x + MLP(LN(x))           # usually GeLU MLP expanding 4× then back
#
# Stack L such blocks → deep ViT.
#
#
# ---------------------------------------------------------------------------
# 3) CLIP — Contrastive Language–Image Pre-training
# ---------------------------------------------------------------------------
#
# Problem: vision models trained only on ImageNet labels know "dog / car / …"
# but not free-form language. Separately, LMs know language but not pixels.
#
# CLIP (Radford et al., 2021): learn a SHARED embedding space where
#   matching (image, text) pairs sit CLOSE together
#   non-matching pairs sit FAR apart
#
# Architecture (two towers):
#
#   Image encoder  f_img  — often a ViT (or ResNet)
#   Text  encoder  f_txt  — Transformer over the caption / prompt
#
#   z_i = normalize( f_img(image_i) )
#   z_t = normalize( f_txt(text_i)  )
#
#   Both mapped to the same D-dim space (L2-normalized → cosine similarity).
#
# Training data: huge batch of N (image, caption) pairs from the internet.
# For each image, its own caption is the POSITIVE; the other N−1 captions
# in the batch are NEGATIVES (and symmetrically for each caption).
#
# At inference (zero-shot classification):
#   1. Encode the image once → z_img
#   2. Encode candidate texts: "a photo of a cat", "a photo of a dog", …
#   3. Pick the text with highest cosine(z_img, z_text)
#   No task-specific fine-tuning required for many categories.
#
# Why it changed the field:
#   - Natural language = flexible label space
#   - Zero-shot / few-shot transfer
#   - Foundation for image–text retrieval, grounding, and many multimodal LMs
#
#
# ---------------------------------------------------------------------------
# 4) THE SYMMETRIC InfoNCE LOSS — "is everywhere"
# ---------------------------------------------------------------------------
#
# InfoNCE (Noise-Contrastive Estimation style) turns similarity into a
# classification problem: "which of these N texts matches this image?"
#
# Similarity matrix for a batch (after L2-norm):
#
#   S_ij = (z_img_i · z_txt_j) / τ
#
#   τ = temperature (learnable or fixed). Smaller τ → sharper distribution.
#
# Image→Text loss (rows): for each image i, true caption is column i
#
#   L_i2t = − (1/N) Σ_i  log  softmax(S)_{i,i}
#         = − (1/N) Σ_i  log  exp(S_ii) / Σ_j exp(S_ij)
#
# Text→Image loss (columns): for each caption j, true image is row j
#
#   L_t2i = − (1/N) Σ_j  log  softmax(Sᵀ)_{j,j}
#
# Symmetric CLIP loss:
#
#   L_CLIP = (L_i2t + L_t2i) / 2
#
# Why BOTH directions?
#   Image→text alone can collapse into "good image features, lazy text encoder"
#   (or vice versa). Symmetry forces BOTH towers to be informative.
#
# "Is everywhere":
#   Same contrastive pattern appears in:
#     - CLIP / ALIGN / SigLIP (vision–language)
#     - SimCLR / MoCo (self-supervised vision: two views of one image)
#     - Dense retrieval (query vs document embeddings)
#     - Audio–text models, etc.
#   Recipe: embed two views → cosine sim → InfoNCE with in-batch negatives.
#
# Intuition with a party:
#   Each photo must find its own caption in a crowd of captions (and each
#   caption must find its photo). Temperature = how harsh the judge is.
#
#
# ---------------------------------------------------------------------------
# BIG PICTURE
# ---------------------------------------------------------------------------
#
#   Image → patches → +pos → Transformer encoder → [CLS]     = ViT
#                                                      │
#                                                      ▼
#                              shared embedding space (cosine)
#                                                      ▲
#   Text  → tokens → +pos → Transformer encoder → pool = Text tower
#                                                      │
#                              trained with Symmetric InfoNCE
#                                                      │
#                                                      ▼
#                              CLIP zero-shot: score image vs prompt texts
#
# Next cells: implement a tiny ViT, then a minimal CLIP-style dual encoder
# and InfoNCE on a toy image–text batch.


In [ ]:
# ViT from Scratch on CIFAR-10 — Setup
# ====================================
#
# TOPIC
# -----
# We will train a tiny Vision Transformer to classify CIFAR-10
# (32×32 colour images, 10 classes: plane, car, bird, …).
#
# Why CIFAR-10 for a teaching ViT?
#   - Small images → few patches → attention is cheap
#   - Fast to download / iterate on CPU or one GPU
#   - Classic benchmark so accuracy is easy to interpret
#
# Pipeline this cell builds:
#   raw image → augment (train) → tensor in [-1, 1] → DataLoader batches
# Later cells: PatchEmbed → Transformer → classify.
#
# Augmentations (train only):
#   RandomCrop + flip → cheap regularization (ViTs overfit small data easily)
#   Normalize mean/std 0.5 → pixels roughly in [-1, 1]
#
#
# CODE MAP
# --------
# imports / seeds / device
# transform_train vs transform_test
# CIFAR10 datasets + DataLoaders (batch 128 train / 256 test)


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Train: light augmentation so the tiny ViT generalizes a bit better
transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),  # shift crop → translation noise
        transforms.RandomHorizontalFlip(),     # left-right symmetry
        transforms.ToTensor(),                 # HWC uint8 → CHW float [0,1]
        transforms.Normalize([0.5] * 3, [0.5] * 3),  # → roughly [-1, 1]
    ]
)

# Test: no randomness — fair, repeatable evaluation
transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ]
)

# download=True pulls CIFAR-10 into ./data on first run
train_set = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
test_set = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

# shuffle train; larger test batch is fine (inference, more throughput)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

print(f"Train images: {len(train_set)} | Test images: {len(test_set)}")
print(f"Classes: {train_set.classes}")


In [ ]:
# Building Blocks — PatchEmbed, Attention, Transformer Block
# ==========================================================
# These three modules are the heart of a ViT:
#
#   Image ──► PatchEmbedding ──► [B, N, D] tokens
#              (+ CLS & pos added later in the full ViT)
#                    │
#                    ▼
#            TransformerBlock × L
#              (LN → MHSA → residual → LN → MLP → residual)
#                    │
#                    ▼
#              sequence of refined patch (+CLS) embeddings
#
# CIFAR-10 defaults below: 32×32 image, patch 4 → N = (32/4)² = 64 patches.


class PatchEmbedding(nn.Module):
    """Split image into non-overlapping patches and project each to dim D.

    Trick: a Conv2d with kernel=stride=patch_size IS patchify + linear embed.
      Input:  [B, 3, 32, 32]
      Output: [B, N, D]  with N = (32/P)²
    """

    def __init__(self, in_channels=3, patch_size=4, embed_dim=192, image_size=32):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        # One filter per embed dim; each looks at one PxP patch (no overlap)
        self.proj = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)  # [B, D, H/P, W/P]  — D channels = patch embeddings
        # Flatten spatial grid → sequence, then [B, D, N] → [B, N, D]
        return x.flatten(2).transpose(1, 2)


class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product self-attention with H heads (concept cell §2).

    For each token: Query/Key/Value → softmax(QKᵀ/√d) V → mix heads with W_O.
    """

    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads  # d_k per head
        # One big linear → split into Q, K, V (efficient fused projection)
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)  # W_O after concat heads
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape  # batch, #tokens, embed_dim
        # [B, N, 3C] → [B, N, 3, H, d] → [3, B, H, N, d]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention weights: [B, H, N, N]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = self.drop(attn.softmax(dim=-1))

        # Weighted values → reassemble heads → output projection
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out(out)


class TransformerBlock(nn.Module):
    """Pre-norm encoder block: x + Attn(LN(x)), then x + MLP(LN(x)).

    Pre-LN (norm before attn/MLP) trains more stably than post-LN for ViTs.
    MLP expands by mlp_ratio (usually 4) with GELU, then projects back.
    """

    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # residual path keeps gradient flow
        x = x + self.mlp(self.norm2(x))
        return x
